# Geodaten für Updates der Quartierspiegel aufbereiten
Datum: 07.10.2021

Mit Colab kann **dieses Jupyter-Notebook interaktiv im Browser** gestartet werden:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DonGoginho/myPy/blob/main/quartierspiegel/quartierspiegel_geodaten_aufbereiten.ipynb)

## Lebensqualitätskarten in den [Quartierspiegeln](https://www.stadt-zuerich.ch/prd/de/index/statistik/publikationen-angebote/publikationen/Quartierspiegel.html)

Für diese Karte werden diverse POIs ausgelesen und als Punkte dargestellt. Damit die Statistik der POIs im Bericht ausgewertet werden kann, hat Stefanie ein R-Skript erstellt, mit dem sie Auswertungen zu den POIs macht. Dazu braucht sie Shapefiles, die vom Geoserver exportiert und historisiert abgespeichert werden.

Bisher habe ich dazu ein Skipt verwendet. Damit dieser Prozess noch transparenter und einfacher wird, erstelle ich nun dieses Notebook.

Die QGIS-Projekte für die Quartierspiegel sind unter 
- `O:\GIS\Sandbox\*JAHR*\quartierspiegel\minimaps` abgelegt.

Dieses Notebook hilft die in der Karte aktiven Layer – die für die Karte vom [**Geoserver**](https://www.stadt-zuerich.ch/geodaten/) als View bezogen sind – als shp zu exportieren.

<!---  
Dieses Notebook verwendet als Vorlage die Webseite «[Kitas in Berlin - Jupyter Notebook](https://juanitorduz.github.io/kitas_berlin/)» 
---!>

## Importiere notwendige Packages:

In [1]:
pip install geopandas altair fiona requests folium mplleaflet contextily seaborn datetime

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import altair as alt
import fiona
import requests
import mplleaflet
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import datetime


## Datenaufbereitung

### Globale Variabeln

Als Zeitvariable berechnen wir erst mal das heutige Datum. Wir können diese Werte als Dateinamen verwenden.

**Steuerungsvariable activate_date**: 
- False: in den Filenamen kommen beim Export KEINE Datenangaben mit
- True: die Exportfiles haben *FILENAMEN_DATUM* im Namen

In [28]:
activate_date = False;


#https://www.programiz.com/python-programming/datetime/strftime
now = datetime.now() # current date and time
year_today = now.strftime("%Y")

if activate_date == True:
    date_today = "_"+now.strftime("%Y-%m-%d")
    #store_o_gis = False
    
else:
    date_today =""
    #store_o_gis = True


In [29]:
print("Datum: "+date_today +" Jahr: "+year_today)

Datum:  Jahr: 2022


### Pfade


In [30]:
gis_ablagepfad = "//szh/ssz/data/GIS/Daten/Vektor/"+year_today+"/quartierspiegel/"
print(gis_ablagepfad )


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/


### Basis Geodaten

Für grundsätzliche räumliche Prüfungen importieren wir hier auch die 
- **Statistischen Quartiere**
- **Stadtgrenze** (aus der SSZ-Geodatenablage)



In [31]:
# Read abt. Geopandas https://geopandas.org/docs/user_guide/io.html
url0 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=adm_statistische_quartiere_map"
gdf_adm_statistische_quartiere_map = gpd.read_file(url0)
print('done')

done


In [32]:
# Read abt. Geopandas https://geopandas.org/docs/user_guide/io.html
pfad_grenze = r"\\szh\ssz\data\GIS\Daten\Vektor\_aktuell\adm_stadtgrenze_a.shp"
gdf_adm_stadtgrenze_lv95 = gpd.read_file(pfad_grenze)

gdf_adm_stadtgrenze_wgs84_all =gdf_adm_stadtgrenze_lv95.to_crs(epsg=4326)

print('done')


done


In [33]:
# Rename columns bei Bedarf
rename_cols = {
    'name': 'bfs_name',
}

columns_to_drop = ['objid']

gdf_adm_stadtgrenze_wgs84 = gdf_adm_stadtgrenze_wgs84_all \
    .copy() \
    .drop(columns_to_drop, axis=1) \
    .rename(columns=rename_cols) \
    

In [34]:
#gdf_adm_stadtgrenze_wgs84

## Geodaten für die Karte aufbereiten

Beziehe die notwendigen Geodaten vom Geoportal und exportiere sie als Shapefiles.


### Alterszentren:

Stefanie muss jedes Jahr mit der DA Alterszentren abklären, welche Alterszentren **ausgeschlossen** werden müssen, da nicht immer alle Bewohnt sind, wegen Umbau oder Ähnlichem. 
Aussderdem gibt es noch solche, die ausserhalb der Stadt liegen. Diese interessieren für die Quartierspiegel nicht und werden ebenfalls herausgefiltert.

#### Step 1: Daten vom Geoportal beziehen und prüfen.

In [36]:
url1 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Alterszentrum?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_alterszentrum_view"

gdf_poi_alterszentrum_view_imp_all = gpd.read_file(url1)

#gdf_poi_alterszentrum_view_imp_all.head(1)
#gdf_poi_alterszentrum_view_imp_all[['name','adresse', 'plz','geometry']]
#gdf_poi_alterszentrum_view_imp_all.info()

##### Einige Pflegezentren gehören offenbar seit 2022 auch dazu... IN ABKLÄRUNG

In [66]:
url_pflegezentren="https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Pflegezentrum?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_pflegezentrum_view"
gdf_poi_pflegezentrum_view = gpd.read_file(url_pflegezentren)

In [67]:
#gdf_poi_pflegezentrum_view[gdf_poi_pflegezentrum_view['name'].str.contains("Gesundheitszentrum"),['adresse']]
gdf_poi_pflegezentrum_view[['name','adresse', 'plz','ort','geometry']].head(2)

,name,adresse,plz,ort,geometry
0,Pflegewohngruppe Felsenrain,Neunbrunnenstrasse 8,8050,Zürich,POINT (8.54420 47.41761)
1,Gesundheitszentrum für das Alter Witikon,Kienastenwiesweg 2,8053,Zürich,POINT (8.60180 47.35509)


#### Step 2: Filtere diejenigen heraus, die nicht benötigt werden

Spatial Join, damit nur jene **innerhalb des Stadtgebiets** vorkommen. Hier sind zwei Möglichkeiten dargestellt, Verschnitt mit Stadtgrenze oder mit den Stadtquartieren. Wichti g ist, dass dabei  auch gerade die Attribute der Geodaten angehängt werden.

In [20]:
gdf_poi_alterszentrum_view_filter_stadtgebiet = gpd.sjoin(gdf_poi_alterszentrum_view_imp_all, gdf_adm_stadtgrenze_wgs84, how='inner', op='within')
#gdf_poi_alterszentrum_view_filter_stadtgebiet = gpd.sjoin(gdf_poi_alterszentrum_view_imp_all, gdf_adm_statistische_quartiere_map, how='inner', op='within')
#gdf_poi_alterszentrum_view_filter_stadtgebiet.info()



In [27]:
#Temportärer Output zur Abstimmung mit dem GUD welche wir ausschliessen sollen und welche nicht...
#gdf_poi_alterszentrum_view_filter_stadtgebiet[['name','adresse', 'plz','ort','geometry']].sort_values('name', ascending =True).to_excel("./output/listen/aktuelle_alterszentren_stadt"+date_today+".xlsx", index=False)

Filtere explizt **vom GUD erwähnte Alterszentren** heraus. (Aktuell sind auch solche angegeben, die ausserhalb des Stadtgebiets liegen. Doppelt gemoppelt ;) )

`
"name" != 'Alterszentrum Buttenau'  AND "name" != 'Alterszentrum Wolfswinkel' 
AND "name" != 'Alterszentrum Sonnenhof' AND "name" != 'Alterszentrum Waldfrieden'  
AND "name" != 'Online Veranstaltung'
`

Am 18.10.2021 wurden ausserdem vom GUD folgende gemeldet, die ausgeschlossen werden sollen:

- Alterszentrum Hardau
- Gästeappartement Alterszentrum Mittelleimbach

In [46]:
#2021:gdf_poi_alterszentrum_view_filtered = gdf_poi_alterszentrum_view_filter_stadtgebiet.query("name !='Alterszentrum Wolfswinkel'  & name !='Alterszentrum Buttenau' & name !='Alterszentrum Sonnenhof' & name !='Gästeappartement Alterszentrum Mittelleimbach'  & name !='Online Veranstaltung' & name !='Alterszentrum Hardau' & name !='Gästeappartement Mittelleimbach'") 
gdf_poi_alterszentrum_view_filtered = gdf_poi_alterszentrum_view_filter_stadtgebiet.query("name !='Alterszentrum Wolfswinkel'  
                                                                                          & name !='Alterszentrum Buttenau' 
                                                                                          & name !='Alterszentrum Sonnenhof' 
                                                                                          & name !='Gästeappartement Alterszentrum Mittelleimbach'  
                                                                                          & name !='Online Veranstaltung' 
                                                                                          & name !='Alterszentrum Hardau' 
                                                                                          & name !='Gästeappartement Mittelleimbach'") 

#gdf_poi_alterszentrum_view_filtered.info()


### Step3: Datensätze exportieren

Speichere eine Liste der aktuell gefilterten Alterszentren als CSV lokal ab.

In [47]:
gdf_poi_alterszentrum_view_filtered[['name','adresse', 'plz','ort','geometry']].to_csv("./output/listen/gefilterte_alterszentren_stadt"+date_today+".csv")
gdf_poi_alterszentrum_view_filtered[['name','adresse', 'plz','ort','geometry']]

,name,adresse,plz,ort,geometry
4,Gästeappartement Gesundheitszentrum für das Al...,Leimbachstrasse 210,8041,Zürich,POINT (8.51497 47.32597)
5,Gesundheitszentrum für das Alter Stampfenbach,Lindenbachstrasse 1,8006,Zürich,POINT (8.53898 47.38690)
6,Gesundheitszentrum für das Alter Langgrüt,Langgrütstrasse 51,8047,Zürich,POINT (8.49968 47.37256)
7,Gesundheitszentrum für das Alter Laubegg,Hegianwandweg 16,8045,Zürich,POINT (8.51348 47.35779)
9,Gesundheitszentrum für das Alter Eichrain,Eichrainstrasse 20,8052,Zürich,POINT (8.54690 47.42882)
10,Gesundheitszentrum für das Alter Mittelleimbach,Leimbachstrasse 210,8041,Zürich,POINT (8.51497 47.32597)
11,Gesundheitszentrum für das Alter Balderen,Klebestrasse 2,8041,Zürich,POINT (8.51298 47.32650)
12,Gesundheitszentrum für das Alter Klus Park,Asylstrasse 130,8032,Zürich,POINT (8.56457 47.36479)
13,Temporäres Gesundheitszentrum für das Alter Tr...,Birmensdorferstrasse 489,8055,Zürich,POINT (8.49717 47.36757)
14,Gesundheitszentrum für das Alter Bullinger-Hardau,Bullingerstrasse 69,8004,Zürich,POINT (8.51055 47.38104)


Wechsle Projektion von WGS84 auf LV95

In [48]:
gdf_poi_alterszentrum_view_filtered_lv95 = gdf_poi_alterszentrum_view_filtered.to_crs(epsg=2056)
#gdf_poi_alterszentrum_view_filtered_lv95['geometry']

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [21]:
print(gis_ablagepfad+"export_poi_alterszentrum_view"+date_today+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_alterszentrum_view.shp


In [50]:
gdf_poi_alterszentrum_view_filtered_lv95.to_file(gis_ablagepfad+"export_poi_alterszentrum_view_"+date_today+".shp", encoding='utf-8')

print('done')
#gdf_poi_alterszentrum_view_filtered_lv95.info()

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1701929876.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_alterszentrum_view_filtered_lv95.to_file(gis_ablagepfad+"export_poi_alterszentrum_view_"+date_today+".shp", encoding='utf-8')


done


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [52]:
gdf_poi_alterszentrum_view_filtered_lv95.to_file("./output/shapefiles/export_poi_alterszentrum_view"+date_today+".shp", encoding='utf-8')
gdf_poi_alterszentrum_view_filtered_lv95.to_file("./output/geopackages/export_poi_alterszentrum_view"+date_today+".gpkg", driver='GPKG')
#gdf_poi_alterszentrum_view_filtered_lv95.plot()

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/4039714076.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_alterszentrum_view_filtered_lv95.to_file("./output/shapefiles/export_poi_alterszentrum_view"+date_today+".shp", encoding='utf-8')


### Alterswohnungen:

Es gibt auch vereinzelt Alterswohnungen ausserhalb der Stadt, daher auch hier noch ein Verschnitt mit der Stadtgrenze.

#### Step 1: Daten vom Geoportal beziehen und prüfen.

In [53]:
url2 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Alterswohnung?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_alterswohnung_view"
gdf_poi_alterswohnungen_view_imp_all = gpd.read_file(url2)
#gdf_poi_alterswohnungen_view_imp_all.info()

#### Step 2: Filtere diejenigen heraus, die nicht benötigt werden

Spatial Join, damit nur jene **innerhalb des Stadtgebiets** vorkommen. Hier sind zwei Möglichkeiten dargestellt, Verschnitt mit Stadtgrenze oder mit den Stadtquartieren. Wichtig ist, dass dabei  auch gerade die Attribute der Geodaten angehängt werden.

In [54]:
gdf_poi_alterswohnungen_view_filter_stadtgebiet = gpd.sjoin(gdf_poi_alterswohnungen_view_imp_all, gdf_adm_stadtgrenze_wgs84, how='inner', op='within')
#gdf_poi_alterswohnungen_view_filter_stadtgebiet.info()

In [55]:
gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95 = gdf_poi_alterswohnungen_view_filter_stadtgebiet.to_crs(epsg=2056)
#gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95['geometry']

#### Step3: Datensätze exportieren


Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [57]:
gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file(gis_ablagepfad+"export_poi_alterswohnung_view_"+date_today+".shp", encoding='utf-8')

print('done')

done


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/2967649901.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file(gis_ablagepfad+"export_poi_alterswohnung_view_"+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [59]:
gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file("./output/shapefiles/export_poi_alterswohnung_view"+date_today+".shp", encoding='utf-8')
gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file("./output/geopackages/export_poi_alterswohnung_view"+date_today+".gpkg", driver='GPKG')
#gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.plot()
print('done')

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/2442419897.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file("./output/shapefiles/export_poi_alterswohnung_view"+date_today+".shp", encoding='utf-8')


done


### Beachvolley (3)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

In [60]:
url3 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_beachvolleyball_view"

typename="poi_beachvolleyball_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#platziere urlX
imp_file = gpd.read_file(url3)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_beachvolleyball_view_wgs84.head(2)

array(['Freibad Zwischen den Hölzern', 'Sportanlage Juchof 1',
       'Josefwiese', 'Schulhaus Letzi', 'Sportzentrum Sihlhölzli',
       'Schulhaus Lachenzelg', 'Schulhaus Stettbach', 'Oberer Letten',
       'Schulhaus Im Lee', 'Schulhaus Buhnrain', 'Freibad Seebach',
       'Sportzentrum Buchlern', 'Freibad Allenmoos',
       'Sportzentrum Fronwald'], dtype=object)

#### Step2: Datensätze exportieren


In [61]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_beachvolleyball_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [63]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_beachvolleyball_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [64]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_beachvolleyball_view.gpkg


### Bikepark (4)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

- url(X) --> Definiere die URL für den GEOJSON-Service
- typename --> Definiere den Layernamen. Kann an diversen Orten verwendet werden
- imp_name --> Leite den importierten Datensatznamen davon ab und ergänze wgs84
- exp_name --> ohne wgs84. soll heissen er ist per default lv95

In [65]:
url4 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_bikepark_view"

typename="poi_bikepark_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url4)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Bike-Park Allmend', 'Pumptrack Heuried'], dtype=object)

#### Step2: Datensätze exportieren


In [66]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [68]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_bikepark_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [69]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_bikepark_view.gpkg


### Eisbahn (5)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

- url(X) --> Definiere die URL für den GEOJSON-Service
- typename --> Definiere den Layernamen. Kann an diversen Orten verwendet werden
- imp_name --> Leite den importierten Datensatznamen davon ab und ergänze wgs84
- exp_name --> ohne wgs84. soll heissen er ist per default lv95

In [70]:
url5 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_eisbahn_view"

typename="poi_eisbahn_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url5)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Sportzentrum Heuried', 'Kunsteisbahn Oerlikon',
       'Kunsteisbahn Dolder'], dtype=object)

#### Step2: Datensätze exportieren


In [71]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [72]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_eisbahn_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [73]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_eisbahn_view.gpkg


### Flussbad (6)

In [74]:
url6 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_flussbad_view"

typename="poi_flussbad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url6)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Frauenbad Stadthausquai', 'Männerbad Schanzengraben',
       'Flussbad Unterer Letten', 'Flussbad Oberer Letten',
       'Flussbad Au-Höngg'], dtype=object)

#### Bug 2021-10-19: Doppelter Eintrag Unterer Letten

In [75]:
imp_file_filtered = imp_file.query("objectid != 7257") 
#imp_file_filtered.info()
#imp_file_filtered

#### Step2: Datensätze exportieren


In [76]:
exp_file = imp_file_filtered.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [77]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_flussbad_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [78]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_flussbad_view.gpkg


### Freibad (7)

In [79]:
url7 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_freibad_view"

typename="poi_freibad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#place urlX
imp_file = gpd.read_file(url7)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Freibad Letzigraben', 'Freibad Dolder',
       'Freibad Zwischen den Hölzern', 'Freibad Allenmoos',
       'Freibad Auhof', 'Freibad Heuried', 'Freibad Seebach'],
      dtype=object)

#### Step2: Datensätze exportieren


In [80]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [81]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_freibad_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [82]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_freibad_view.gpkg


### Fussballplätze (8)

In [83]:
url8 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_fussballplatz_view"

typename="poi_fussballplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url8)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Rasensportanlage Katzenbach', 'Rasensportanlage Juchhof 2',
       'Rasensportanlage Allmend Brunau', 'Sportzentrum Buchlern',
       'Rasensportanlage Forrenweid', 'Sportzentrum Hardhof',
       'Rasensportanlage Juchhof 1', 'Sportzentrum Utogrund',
       'Sportzentrum Eichrain', 'Sportzentrum Fronwald',
       'Sportzentrum Heerenschürli', 'Sportzentrum Witikon',
       'Rasensportanlage Neudorf', 'Sportzentrum Heuried',
       'Rasenspielfeld Liguster'], dtype=object)

#### Step2: Datensätze exportieren


In [84]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [85]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_fussballplatz_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [86]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_fussballplatz_view.gpkg


### Gemeinschaftszentrum (9)

In [87]:
url9 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Gemeinschaftszentrum?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_gemeinschaftszentrum_view"

typename="poi_gemeinschaftszentrum_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url9)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['GZ Schindlergut', 'GZ Heuried', 'GZ Hirzenbach', 'GZ Hottingen',
       'GZ Leimbach', 'GZ Loogarten', 'GZ Riesbach', 'GZ Seebach',
       'GZ Wipkingen', 'GZ Witikon', 'GZ Leimbach - Standort Manegg',
       'GZ Wollishofen - Standort Albisstrasse',
       'GZ Heuried - Standort Manesse', 'GZ Buchegg', 'GZ Grünau',
       'GZ Höngg', 'GZ Wollishofen, Standort Neubühl', 'GZ Affoltern',
       'GZ Bachwiesen', 'GZ Wollishofen', 'GZ Oerlikon',
       'GZ Höngg - Standort Rütihof',
       'GZ Hottingen - Standort Klosbachstrasse',
       'GZ Loogarten - Standort Badenerstrasse'], dtype=object)

#### Step2: Datensätze exportieren


In [88]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [89]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_gemeinschaftszentrum_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [90]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_gemeinschaftszentrum_view.gpkg


### Hallenbad (10)

In [91]:
url10 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_hallenbad_view"

typename="poi_hallenbad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url10)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Hallenbad Altstetten', 'Hallenbad Bläsi', 'Hallenbad Bungertwies',
       'Hallenbad City', 'Wärmebad Käferberg', 'Hallenbad Leimbach',
       'Hallenbad Oerlikon'], dtype=object)

#### Step2: Datensätze exportieren


In [92]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [93]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_hallenbad_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [94]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_hallenbad_view.gpkg


### Jugendtreff (11)

In [95]:
url11 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Jugendtreff?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_jugendtreff_view"

typename="poi_jugendtreff_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url11)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['OJA Affoltern', 'Verein Jugendtreff Kreis 4',
       'OJA Kreis 9 & Hard', 'OJA Wollishofen-Leimbach',
       'OJA Kreis 5/planet 5', 'OJA Kreis 6 & Wipkingen', 'OJA Oerlikon',
       'OJA Schwamendingen',
       'OJA Kreis6&Wipkingen - Standort Bhf.Wipkingen',
       'Jugendkulturhaus Dynamo', 'Verein Jugendarbeit Auzelg (c/o Hort)',
       'OJA Kreis 3 & 4', 'OJA Wollishofen&Leimbach Standort Leimbach',
       'Mundwerk Kulturbiotop', 'OJA Kreis 9 & Hard Standort Hardstrasse'],
      dtype=object)

#### Step2: Datensätze exportieren


In [96]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [97]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_jugendtreff_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [98]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_jugendtreff_view.gpkg


### Park (12)

In [99]:
url12 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Park?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_park_view"

typename="poi_park_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url12)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Zeughaushof', 'Park Pfingstweid', 'Arboretum', 'Tiefenbrunnen',
       'Riedgraben', 'Klopstockwiese', 'Wahlenpark', 'Mattenhof',
       'Felsenrainanlage', 'Schärrerwiese', 'Leutschenpark', 'Birchwald',
       'Heiligfeld', 'Langgrütweg', 'Süssleren',
       'Alter Botanischer Garten', 'Am Katzenbach', 'Schindlergut',
       'Beckenhof', 'Irchelpark', 'Bullingerhof', 'Fritschiwiese',
       'Bachwiesenpark', 'Allmend Brunau', 'Allmend Stettbach',
       'Oerliker Park', 'Waldgarten', 'Saatlen-Grünzug',
       'Schörli-Grünzug', 'Botanischer Garten', 'GZ Riesbach',
       'Hambergsteig', 'Platzspitz', 'Ruggächeranlage', 'GZ Buchegg',
       'Alfred-Altherr-Terrasse', 'Herzogenmühle-Grünzug', 'Glattwiesen',
       'GZ Hirzenbach', 'Blatterwiese', 'Villa Schönberg', 'Seefeldquai',
       'Buchholz', 'Im Gut', 'Sihlhölzli', 'Villa Bleuler',
       'Neumünsteranlage', 'Patumbah-Park', 'Villa Tobler', 'Belvoirpark',
       'GZ Grünau', 'Waidberg', 'Wipkingerpark', 'Landenberganlage'

#### Step2: Datensätze exportieren


In [100]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [101]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_park_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [102]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_park_view.gpkg


### Picknick (13)

In [103]:
url13 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Erholungs__und_Sporteinrichtungen?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_picknickplatz_view"

typename="poi_picknickplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url13)

imp_file.info()
imp_file['name'].unique()


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 57 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   id                           99 non-null     object  
 1   adr_inter                    0 non-null      object  
 2   adresse                      0 non-null      object  
 3   adrzus_int                   0 non-null      object  
 4   anlageelemente               47 non-null     object  
 5   behindertenparkplatz         0 non-null      object  
 6   bemerkung                    0 non-null      object  
 7   bereich                      0 non-null      object  
 8   ccmail                       0 non-null      object  
 9   da                           99 non-null     object  
 10  datum                        99 non-null     object  
 11  datum_cms                    0 non-null      object  
 12  dep                          99 non-null     object  
 13 

array(['Tamboureneiche', 'Entlisbergkopf', 'Rüteliweg', 'Bergholz',
       'Kappeli', 'Schachtisch', 'Tobelholzweg', 'Rumpelhalde',
       'Rötelterrasse', 'Unterer Spielplatz', 'Eichhörnliweg',
       'Oberer Spielplatz', 'Skatmatt', 'Rondell', 'Reservoir Käferberg',
       'Käferberg-Weiher', 'Waidbergweg', 'Zweifelseiche', 'Elefant',
       'Stöckentobelhütte', 'Degenried', 'Blaumeise', 'Biberlinsterrasse',
       'Im Schlagbach', 'Dunkelhölzli', 'Reservoir Lyren', 'Frauenmatt',
       'Cherigatter', 'Grossmutter', 'Hasenrain',
       'Oetenbacherholzstrasse', 'Loorenkopf', 'Boxerbrunnen',
       'Thalwilerbänkli', 'Lindenweg, Altes Reservoir', 'Waid',
       'ARA Adliswil', 'Entlisberghütte', 'Hans Rölliweg', 'Känzeli',
       'Untere Forsthausstrasse', 'Tscharli Platz', 'Oerliker Wäldli',
       'Friesenberg', 'Kolbenhof', 'Milan', 'Wasserrad', 'Spinne',
       'Hasenrainhütte', 'Hurdäckerstrasse', 'Höckler', 'Hochwacht',
       'ARA  Adliswil', 'Buchenweg', 'Burgwies', 'Mädikon',

#### Step2: Datensätze exportieren


In [104]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [105]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_picknickplatz_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [106]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_picknickplatz_view.gpkg


### Quartiertreffhaus (14)

In [107]:
url14 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Quartiertreff?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_quartiertreffhaus_view"

typename="poi_quartiertreffhaus_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url14)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Quartiertreff Hirslanden', 'Quartierraum Zentralstrasse',
       'Quartierräume Bullinger', 'Quartiertreff Zehntenhaus',
       'Quartierhaus Schütze (Mehrzweckraum)',
       'Quartiertreff Altstadthaus', 'Spielbaracke',
       'Quartierzentrum Bäckeranlage', 'Schopf - Verein Hochneun',
       'Quartiertreff Waldgarten', 'Quartierzentrum Schütze',
       'Quartiertreff Hüttenkopf', 'Quartierbaracke Auzelg', 'Stadtgeiss',
       'Quartierraum Viadukt', 'Kulturbahnhof Affoltern, KuBaA',
       'Kiosk Josefwiese', 'Quartierräume Hardau', 'Solino',
       'Quartierhaus Kreis 6', 'Quartiertreff Enge', 'Quartierhof Wynegg',
       'Zentrum Karl der Grosse', 'Quartierraum Fritschiwiese',
       'Quartiertreff Fluntern'], dtype=object)

#### Step2: Datensätze exportieren


In [108]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [109]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_quartiertreffhaus_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [110]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_quartiertreffhaus_view.gpkg


### Seebad (15)

In [112]:
url15 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_seebad_view"

typename="poi_seebad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url15)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Strandbad Wollishofen', 'Strandbad Tiefenbrunnen', 'Seebad Enge',
       'Seebad Katzensee', 'Seebad Utoquai', 'Strandbad Mythenquai'],
      dtype=object)

#### Step2: Datensätze exportieren


In [113]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [114]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_seebad_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [115]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_seebad_view.gpkg


### Skateranlage (16)

In [116]:
url16 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_skateranlage_view"

typename="poi_skateranlage_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url16)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Freestyleanlage Leimbach', 'Freestyle-Halle Werdhölzli',
       'Freestyleanlage Buchlern', 'Freestyleanlage Heerenschürli',
       'Freestyleanlage Hohlstrasse', 'Freestyleanlage Fronwald',
       'Freestyleanlage Kornhausbrücke', 'Freestyle-Park Allmend'],
      dtype=object)

#### Step2: Datensätze exportieren


In [117]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [118]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_skateranlage_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [119]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_skateranlage_view.gpkg


### Spielplatz (17)

In [129]:
url17 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Spielgeraetekataster?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_spielplatz_view"

typename="poi_spielplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url17)

#imp_file.info()
imp_file['name'].unique()
#imp_file.head()


array(['Manessestrasse / Weststrasse',
       'GZ Seebach                                                            ',
       'Bertahof                                                              ',
       'Bauspielplatz Rütihütten                                              ',
       'Klingenpark                                                           ',
       'Saatlen-Grünzug                                                       ',
       'Hallenbad Oerlikon                                                    ',
       'Rieteranlage                                                          ',
       'Opfikonstrasse', 'Frauental Tennisanlag', 'Bachwiesen Grünzug',
       'Quartierpark Schütze (A) ',
       'GZ Hottingen                                                          ',
       'Hohenstein                                                            ',
       'Hasenrain                                                             ',
       'Louis Häfliger Park                

#### Step2: Datensätze exportieren


In [130]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [131]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_spielplatz_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [132]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_spielplatz_view.gpkg


### Sporthalle (18)

In [133]:
url18 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_sporthalle_view"

typename="poi_sporthalle_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url18)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Schul- und Sportanlage Leimbach', 'Sportzentrum Fronwald',
       'Dreifachhalle Im Birch', 'Dreifachhalle Hofacker',
       'Schul- und Sportanlage Döltschi',
       'Schul- und Sportanlage Schauenberg', 'ASVZ Rämibühl',
       'Sport Center Hönggerberg', 'Schul- und Sporthalle Hirzenbach',
       'Dreifachhalle Hardau', 'Sportzentrum Sihlhölzli',
       'Schul- und Sportanlage Leutschenbach', 'Saalsporthalle',
       'Dreifachhalle Blumenfeld', 'Sportzentrum Utogrund',
       'Schul- und Sportanlage Falletsche',
       'Schul- und Sportanlage Buchwiesen',
       'Schul- und Sportanlage Stettbach',
       'Schul- und Sportanlage Apfelbaum', 'Sportzentrum Josef',
       'Einfachhalle Riesbach'], dtype=object)

#### Step2: Datensätze exportieren


In [134]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [136]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_sporthalle_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [137]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_sporthalle_view.gpkg


### Stadion (19)

In [138]:
url19 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_stadion_view"

typename="poi_stadion_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url19)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Stadion Letzigrund'], dtype=object)

#### Step2: Datensätze exportieren


In [139]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [140]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_stadion_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [141]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_stadion_view.gpkg


### Tennisplatz (20)

In [142]:
url20 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_tennisplatz_view"

typename="poi_tennisplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url20)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Tennisanlage Frauental', 'Tennisanlage Ueberlandstrasse',
       'Tennisanlage Hardhof', 'Tennisanlage Buchlern',
       'Tennisanlage Eichrain', 'Tennisanlage Fronwald',
       'Tennisanlage Lengg', 'Tennisanlage Mythenquai'], dtype=object)

#### Step2: Datensätze exportieren


In [143]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [144]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_tennisplatz_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [145]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_tennisplatz_view.gpkg


### Waldhütte (21)

In [146]:
url21 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Erholungs__und_Sporteinrichtungen?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_waldhuette_view"

typename="poi_waldhuette_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url21)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Waldhütte Escherhöhe', 'Waldhüsli', 'Moosholz', 'Huebbach',
       'Hasenrain', 'Forsthaus Frauenmatt', 'Loorenkopf, Adlisberg',
       'Adlisberghütte', 'Köhlerstrasse', 'Buschberg',
       'Hohensteinstrasse', 'Annaburg', 'Elefant', 'Stöckentobelhütte',
       'Tscharli Platz', 'Waldhütte Ziegelhöhe'], dtype=object)

#### Step2: Datensätze exportieren


In [147]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [148]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_waldhuette_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [149]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_waldhuette_view.gpkg


### Zürifit (22)

In [150]:
url22 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_zuerifit_view"

typename="poi_zuerifit_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url22)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Zürifit Irchelpark', 'Zürifit Hardhof',
       'Zürifit Neunbrunnenstrasse', 'Zürifit Brunau'], dtype=object)

#### Step2: Datensätze exportieren


In [151]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [152]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_zuerifit_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [153]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_6096/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_zuerifit_view.gpkg


### ----------------------------------------
### Vorlage (X)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

- url(X) --> Definiere die URL für den GEOJSON-Service
- typename --> Definiere den Layernamen. Kann an diversen Orten verwendet werden
- imp_name --> Leite den importierten Datensatznamen davon ab und ergänze wgs84
- exp_name --> ohne wgs84. soll heissen er ist per default lv95

In [156]:
urlX = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=X"

typename="X"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url4)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Bike-Park Allmend', 'Pumptrack Heuried'], dtype=object)

#### Step2: Datensätze exportieren


In [157]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [158]:
#exp_file.to_file(gis_ablagepfad+"export_"+typename+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2021/quartierspiegel/export_X.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [490]:
exp_file.to_file("./output/shapefiles/"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackage/"+typename+date_today+".gpkg", driver='GPKG')
#gdf_poi_bikepark_view.plot()
print("GPKG exportiert nach: "+"./output/geopackage/"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_9792/2296635697.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/"+typename+date_today+".shp", encoding='utf-8')
